## Hadoop: Writing and Reading Data with HDFS

This notebook demonstrates how to interact with Hadoop's Distributed File System (HDFS) using Python. We will cover the fundamental operations of writing data to HDFS and reading data from HDFS.


#### 1. Connect to HDFS

Check connectivity to the HDFS system and create a client object.

In [ ]:
from hdfs import InsecureClient
try:
    client = InsecureClient('http://localhost:14000', user='root')
    print("Connected!")
except Exception as e:
    print(e)

In [ ]:
# Verify proxy is configured
assert client.list('/') is not None
print('Proxy OK')

#### 2. Check HDFS folders and files

In [ ]:
files = client.list('/')
print(files)

#### 3. Download and save the sample data file to your local machine.

In [ ]:
import urllib.request
import zipfile
from os import remove
import os

os.makedirs('../temp', exist_ok=True)
url = 'https://www.kaggle.com/api/v1/datasets/download/chaitanyahivlekar/large-movie-dataset'
urllib.request.urlretrieve(url,'../temp/movies.zip')

with zipfile.ZipFile('../temp/movies.zip', 'r') as zip_ref:
    zip_ref.extractall('../temp/')

remove('../temp/movies.zip')

#### 4. Copy the file to the HDFS

Using the proxy (HttpFS, port 14000), uploads go through a proxy
that resolves DataNode hostnames internally — no subprocess needed.

Check on browser [localhost:9870](http://localhost:9870/explorer.html#/)

In [ ]:
client.upload(hdfs_path='movies_data.csv', local_path='../temp/movies_dataset.csv', overwrite=True, permission=775)

#### 5. Read data from HDFS

Read data from HDFS into a pandas dataframe.

In [ ]:
import pandas as pd

with client.read(hdfs_path='/user/root/movies_data.csv', encoding='utf8') as file:
  df = pd.read_csv(file)

df